In [152]:
class T_symb:
    def __init__(self,heis_dim,pickled_ad=False):
        self.heis_dim=heis_dim
        
        self.basis_strs=['Y','H','E','X']
        for i in range(1,heis_dim):
            self.basis_strs.append('e%d'%i)
        self.basis_strs.append('N')
        
        # Weights
        wght_list=[1,0,0,-1]
        for i in range(4,len(self.basis_strs)-1):
            wght_list.append(-i+3)
        wght_list.append(-heis_dim)
        
        # Bases
        self.basis=[T_symb_basis_elt(self.basis_strs[i],wght_list[i],self) 
                    for i in range(len(self.basis_strs))]
        self.gl2_basis=self.basis[0:4]
        self.heis_basis=self.basis[4:len(self.basis)]
        self.V_basis=self.heis_basis[0:len(self.heis_basis)-1]
        
        self.ad_dict={}
        self.set_ad_dict(pickled_ad)
        
    def set_ad_dict(self,pickled_ad=False):
        k=len(self.basis)

        Y=self.gl2_basis[0]
        H=self.gl2_basis[1]
        E=self.gl2_basis[2]
        X=self.gl2_basis[3]
        N=self.heis_basis[len(self.heis_basis)-1]
        ## Set ad_dicts
        #  First, set ad_dict for each T_symb_basis_elt

        for A in self.gl2_basis:
            E.ad_dict[str(A)]=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)+1):
            E.ad_dict[str(self.V_basis[i-1])]=self.V_basis[i-1]
        E.ad_dict['N']=2*N

        X.ad_dict['Y']=H
        X.ad_dict['H']=-2*X
        X.ad_dict['E']=T_symb_elt([0]*k,self)
        X.ad_dict['X']=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)):
            X.ad_dict[str(self.V_basis[i-1])]=self.V_basis[i]
        X.ad_dict[str(self.V_basis[len(self.V_basis)-1])]=T_symb_elt([0]*k,self)
        X.ad_dict['N']=T_symb_elt([0]*k,self)

        Y.ad_dict['Y']=T_symb_elt([0]*k,self)
        Y.ad_dict['H']=2*Y
        Y.ad_dict['E']=T_symb_elt([0]*k,self)
        Y.ad_dict['X']=-H
        Y.ad_dict[str(self.V_basis[0])]=T_symb_elt([0]*k,self)
        for i in range(2,len(self.V_basis)+1):
            Y.ad_dict[str(self.V_basis[i-1])]=(i-1)*(self.heis_dim-i)*self.V_basis[i-2]
        Y.ad_dict['N']=T_symb_elt([0]*k,self)

        H.ad_dict['Y']=-2*Y
        H.ad_dict['H']=T_symb_elt([0]*k,self)
        H.ad_dict['E']=T_symb_elt([0]*k,self)
        H.ad_dict['X']=2*X
        for i in range(1,len(self.V_basis)+1):
            H.ad_dict[str(self.V_basis[i-1])]=(2*i-self.heis_dim)*self.V_basis[i-1]
        H.ad_dict['N']=T_symb_elt([0]*k,self)

        for i in range(1,len(self.V_basis)+1):
            if i>=2: self.V_basis[i-1].ad_dict['Y']=-(i-1)*(self.heis_dim-i)*self.V_basis[i-2]
            else: self.V_basis[i-1].ad_dict['Y']=T_symb_elt([0]*k,self)
            self.V_basis[i-1].ad_dict['H']=-(2*i-self.heis_dim)*self.V_basis[i-1]
            self.V_basis[i-1].ad_dict['E']=-self.V_basis[i-1]
            if i<=self.heis_dim-2: self.V_basis[i-1].ad_dict['X']=-self.V_basis[i]
            else: self.V_basis[i-1].ad_dict['X']=T_symb_elt([0]*k,self)
            for j in range(1, len(self.V_basis)+1):
                if i+j==self.heis_dim: self.V_basis[i-1].ad_dict[str(self.V_basis[j-1])]=(-1)**i*N
                else: self.V_basis[i-1].ad_dict[str(self.V_basis[j-1])]=T_symb_elt([0]*k,self)
            self.V_basis[i-1].ad_dict['N']=T_symb_elt([0]*k,self)

        N.ad_dict['Y']=T_symb_elt([0]*k,self)
        N.ad_dict['H']=T_symb_elt([0]*k,self)
        N.ad_dict['E']=-2*N
        N.ad_dict['X']=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)+1):
            N.ad_dict[str(self.V_basis[i-1])]=T_symb_elt([0]*k,self)
        N.ad_dict['N']=T_symb_elt([0]*k,self)
    
    def elt(self,vec_rep):
        return T_symb_elt(vec_rep,self)
    
    def ad(self,elt1,elt2):
        '''elt1, elt2: a T_symb_elt objects
        returns: a T_symb_elt object representing ad(se,c)'''

        if elt1==0 or elt2==0:
            return T_symb_elt([0]*len(self.basis),self)

        result=T_symb_elt([0]*len(self.basis),self)
        for j in range(len(self.basis)):
            if elt2.vec_rep[j]!=0:
                for i in range(len(self.basis)):
                    if elt1.vec_rep[i]!=0: 
                        coeff=elt1.vec_rep[i]*elt2.vec_rep[j]
                        result+=coeff*self.basis[i].ad_dict[self.basis_strs[j]]
        return result

In [153]:
# If this class is modified, the pickling of ad dicts should be re-executed

# To do: check equality of parents for many methods
class T_symb_basis_elt:
    def __init__(self,str_rep,wght,parent):
        self.heis_dim=parent.heis_dim
        self.parent=parent
        self.heis_dim=parent.heis_dim
        self.str_rep=str_rep
        
        self.vec_rep=[0]*(self.heis_dim+4)
        basis_str_list=['Y','H','E','X']
        for i in range(1,self.heis_dim):
            basis_str_list.append('e%d'%i)
        basis_str_list.append('N')
        self.vec_rep[basis_str_list.index(str_rep)]=1
        
        self.wght=wght
        self.ad_dict={}
        self.dual_ad_dict={}
        self.cochain_ad_dicts=[{},{},{},{}]
        self.ext_ad_dicts=[{},{},{},{}]
    
    def __eq__(self,other):
        if other==0:
            return False
        return self.vec_rep==other.vec_rep
        
    def __str__(self):
        return self.str_rep
    
    def __repr__(self):
        return self.str_rep
    
    def __lt__(self,other):
        return self.parent.basis.index(self)<self.parent.basis.index(other)
    
    def __gt__(self,other):
        return self.parent.basis.index(self)>self.parent.basis.index(other)
    
    def __le__(self,other):
        return self.parent.basis.index(self)<=self.parent.basis.index(other)

    def __ge__(self,other):
        return self.parent.basis.index(self)>=self.parent.basis.index(other)
    
    def __add__(self,other):
        if type(other)==type(self):
            result=[0]*len(self.parent.basis)
            result[self.parent.basis.index(self)]+=1
            result[self.parent.basis.index(other)]+=1
            return T_symb_elt(result,self.parent)
        result=other.vec_rep
        result[self.parent.basis.index(self)]+=1
        return T_symb_elt(result,self.parent)
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=-1
        return T_symb_elt(result,self.parent)
    
    def __sub__(self,other):
        if type(other)==type(self):
            return self+(-other)
        result=[-A for A in other.vec_rep]
        result[self.parent.basis.index(self)]+=1
        return T_symb_elt(result,self.parent)
    
    def __mul__(self,other):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=other
        return T_symb_elt(result,self.parent)
    
    def __rmul__(self,other):
        return(self*other)
    
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d

In [154]:
class T_symb_elt:
    
    def __init__(self,vec_rep,parent):
        '''vec_rep: a list of length len(self.basis) with integer entries'''
        self.parent=parent
        self.basis=parent.basis
        self.vec_rep=vec_rep
        self.heis_dim=parent.heis_dim
    
    def __str__(self):
        if self.vec_rep==[0]*len(self.basis):
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(self.basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(self.basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(self.basis[cntr])
            cntr+=1
        for i in range(cntr,len(self.basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(self.basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(self.basis[i])
            elif self.vec_rep[i]!=0:
                result+=' + '+str(self.vec_rep[i])+'*'+str(self.basis[i])
        return result

    def __repr__(self):
        return str(self)
    
    def __eq__(self,other):
        if other==0:
            return self.vec_rep==[0]*(len(self.basis))
        return self.vec_rep==other.vec_rep
    
    def __neg__(self):
        return(T_symb_elt([-A for A in self.vec_rep],self.parent))
    
    def __add__(self,other):
        return T_symb_elt([self.vec_rep[i]+other.vec_rep[i] 
                           for i in range(len(self.basis))],self.parent)   
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return T_symb_elt([other*A for A in self.vec_rep],self.parent)
    
    def __rmul__(self,other):
        return self*other
    
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d